# EE708 Project 3: PM2.5 spatial enhancement over India
**Group 1:** Harsh Meshram, Chandranil Das, Dhrubojyoti Saha, Dineshwar Okram, Tanuj Singh. IIT Kanpur.

Run from this extracted submission folder. The executed notebook starts from the supplied processed daily CAMS table. Original raw-data preprocessing and mapping scripts are preserved in `src/`; rerunning those requires the original raw NetCDF files and the region/daily NetCDF inputs.

The reference is CAMS PM2.5, not station measurements. Held-out locations cover all years; this is spatial evaluation, not future forecasting.

In [1]:
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from model_io import load_model, save_model
from evaluate import metrics
FEATURES = ['aod','u10','v10','t2m','d2m','sp','lat','lon','month']
print('Libraries imported')

Libraries imported


## 1. Preprocessing and data checks
The original pipeline concatenates 60 monthly files, averages 3-hourly values by day, converts PM2.5 from kg/m3 to micrograms/m3, temperatures from kelvin to Celsius and pressure from Pa to hPa. The India mask and regional labels are supplied inputs. Daily averaging assumes complete 3-hourly coverage; the processed table alone cannot establish that coverage.

In [2]:
data = pd.read_csv('data/pm25_daily.csv', parse_dates=['date'])
cells = pd.read_csv('data/cells.csv')
assert not data.duplicated(['date','lat','lon']).any()
assert not data.isna().any().any()
print('Rows:', len(data), 'Cells:', len(cells), 'Days:', data.date.nunique())
print('Period:', data.date.min(), 'to', data.date.max())
print('PM2.5 range:', data.pm25.min(), data.pm25.max())
print(data.pivot_table(index='region',columns='season',values='pm25').round(2).to_string())

Rows: 902538 Cells: 494 Days: 1827
Period: 2020-01-01 00:00:00 to 2024-12-31 00:00:00
PM2.5 range: 0.00087396754 2700.274
season     monsoon  post_monsoon  pre_monsoon  winter
region                                               
central      37.51         80.72        55.43   84.71
himalaya     29.82         46.81        37.44   51.75
igp          77.35        207.12       116.52  217.92
northeast    22.48         43.86        82.30   68.42
northwest    67.44         84.95        69.19   85.11
south        30.24         62.22        49.87   63.38


## 2. Spatial block split
Each 0.75-degree cell is assigned to a 5 by 5 block. Blocks satisfying `(block_row + 2*block_col) % 5 == 0` are held out. No day of a test cell is used in fitting. Boundary test cells can be close to training cells; this is not a uniform distance buffer. No hyperparameter tuning on the held-out scores is performed here.

In [3]:
expected = (cells.block_row + 2*cells.block_col) % 5 == 0
assert np.array_equal(expected, cells['set'].eq('test'))
data = data.merge(cells[['lat','lon','set']], on=['lat','lon'], validate='many_to_one')
train = data[data['set']=='train'].copy()
test = data[data['set']=='test'].copy()
print(pd.crosstab(cells.region,cells['set']).to_string())
print('Training rows:',len(train),'Held-out rows:',len(test))

set        test  train
region                
central      42    108
himalaya      9     28
igp          22     62
northeast     7     36
northwest    10     73
south        22     75
Training rows: 697914 Held-out rows: 204624


## 3. Models
Linear regression estimates one global linear relationship. The random forest averages 100 regression trees with minimum 10 samples per leaf and seed 42. IDW uses the 8 nearest training cells on the same day with inverse squared distance weights; distance in degrees is an approximation.

The default below loads the submitted trained forest. Set `RETRAIN=True` to refit from the training cells. Refitting can vary with library version and takes longer; it does not overwrite the submitted model.

In [4]:
RETRAIN = False
if RETRAIN:
    model = RandomForestRegressor(n_estimators=100,min_samples_leaf=10,n_jobs=-1,random_state=42)
    model.fit(train[FEATURES],train.pm25)
    save_model(model,'Model_retrained.h5')
else:
    model = load_model('Model.h5')
print(model)
linear = LinearRegression().fit(train[FEATURES],train.pm25)
print('Linear regression fitted on training cells only')

RandomForestRegressor(min_samples_leaf=10, n_jobs=-1, random_state=42)
Linear regression fitted on training cells only


In [5]:
pm = data.pivot(index='date',columns=['lat','lon'],values='pm25')
trcells = cells[cells['set']=='train']
idw_records = []
for cell in cells[cells['set']=='test'].itertuples():
    distance = np.sqrt((trcells.lat-cell.lat)**2 + (trcells.lon-cell.lon)**2)
    nearest = distance.nsmallest(8)
    weights = (1/nearest**2); weights /= weights.sum()
    neighbours = trcells.loc[nearest.index]
    pred = pm[list(zip(neighbours.lat,neighbours.lon))].to_numpy() @ weights.to_numpy()
    idw_records.append(pd.DataFrame({'date':pm.index,'lat':cell.lat,'lon':cell.lon,'idw':pred}))
test = test.merge(pd.concat(idw_records),on=['date','lat','lon'],validate='one_to_one')
test['linear'] = linear.predict(test[FEATURES])
test['random_forest'] = model.predict(test[FEATURES])
print('All three held-out predictions computed')

All three held-out predictions computed


## 4. Training and held-out performance
Training scores are resubstitution scores, not independent validation. The held-out spatial test is used as the required validation assessment. All metrics below retain extreme values and negative linear predictions.

In [6]:
training_pred = model.predict(train[FEATURES])
print('RF training:',metrics(train.pm25,training_pred))
print(pd.DataFrame({m:metrics(test.pm25,test[m]) for m in ['idw','linear','random_forest']}).T.round(6).to_string())
for group in ['region','season']:
    rows=[{group:k,'method':m,**metrics(d.pm25,d[m])} for k,d in test.groupby(group) for m in ['idw','linear','random_forest']]
    print('\n'+group)
    print(pd.DataFrame(rows).pivot(index=group,columns='method',values='R2').round(4).to_string())

RF training: {'RMSE': 18.845496579135702, 'MAE': 8.474647137648226, 'R2': 0.9147895772623355, 'bias': 0.0245618578300186}
                    RMSE        MAE        R2       bias
idw            35.339735  20.432134  0.683275  -6.750651
linear         46.721478  32.138917  0.446409 -12.876700
random_forest  31.888101  17.818622  0.742123  -5.583709

region
method        idw  linear  random_forest
region                                  
central    0.6963  0.4361         0.7523
himalaya   0.4572  0.4931         0.7843
igp        0.5557  0.2860         0.5380
northeast -0.7401  0.1493         0.7378
northwest  0.8047 -0.0485         0.7476
south      0.4506 -1.7971         0.4742

season
method           idw  linear  random_forest
season                                     
monsoon       0.7845  0.3550         0.8050
post_monsoon  0.6355  0.4384         0.7147
pre_monsoon   0.5357  0.1970         0.6005
winter        0.6104  0.3679         0.6903


## 5. Model summary and efficiency
Tree nodes and leaves describe forest complexity. They are not neural-network parameter counts. Prediction time depends on hardware and includes the batch prediction call; it excludes reading data.

In [7]:
start = time.perf_counter()
_ = model.predict(test[FEATURES])
elapsed = time.perf_counter()-start
print('Trees:',len(model.estimators_))
print('Total nodes:',sum(t.tree_.node_count for t in model.estimators_))
print('Total leaves:',sum(t.tree_.n_leaves for t in model.estimators_))
print('Maximum depth:',max(t.tree_.max_depth for t in model.estimators_))
print('HDF5 bytes:',Path('Model.h5').stat().st_size)
print('Held-out prediction seconds:',round(elapsed,3))
print(pd.Series(model.feature_importances_,index=FEATURES).sort_values(ascending=False).to_string())
print('Negative linear predictions:',int((test.linear<0).sum()))

Trees: 100
Total nodes: 6886686
Total leaves: 3443393
Maximum depth: 39
HDF5 bytes: 182929735
Held-out prediction seconds: 0.381
aod      0.305218
sp       0.193309
t2m      0.136584
d2m      0.102273
lat      0.088157
lon      0.079420
u10      0.037041
v10      0.029150
month    0.028848
Negative linear predictions: 14108


## 6. Spatial enhancement and limits
The supplied figure compares full-field CAMS PM2.5, bilinear interpolation of CAMS PM2.5, and RF predictions from bilinearly interpolated predictors. These map comparisons are descriptive. The bilinear map uses the full reference field, whereas held-out IDW uses training cells only. The 0.25-degree output has no independent fine-scale truth and introduces no new high-resolution predictors.

![2024 maps](figures/map_2024_mean.png)

Unweighted grid means (micrograms/m3): CAMS 72.80, bilinear 72.37, RF 71.54. Mean absolute RF-bilinear difference: 5.41. Similar national means do not prove local consistency or accuracy.

## 7. Command-line evaluation
`python evaluate.py --dataset data/pm25_daily.csv --cells data/cells.csv`

The HDF5 model contains numeric tree state and metadata, not a renamed joblib or Keras model. Use the supplied loader and pinned requirements. References: CAMS EAC4 dataset; Breiman (2001), Random Forests; scikit-learn documentation.

In [8]:
# Optional regeneration of the original step-6 maps using the HDF5 model.
# The supplied original script is adapted in memory; no source file is overwritten.
RUN_MAPS = False
if RUN_MAPS:
    mapping_source = Path('src/step6_maps.py').read_text()
    mapping_source = mapping_source.replace(
        'joblib.load("results/step4/random_forest.joblib")',
        'load_model("Model.h5")'
    )
    exec(compile(mapping_source, 'step6_maps_hdf5', 'exec'), globals())
else:
    print('Set RUN_MAPS=True to regenerate maps from data/pm25_daily.nc.')


Set RUN_MAPS=True to regenerate maps from data/pm25_daily.nc.
